# Simulation-Based Inference (SBI) with variational inference
We'll go through some examples of Bayesian inference with SBI using variational inference to estimate the joint $p(\theta,\mathcal{D})$. We'll use the same examples as in the ABC demo and use `pytorch`.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import corner 

In [ ]:
torch.manual_seed(0)

# Example 0
Let's return to the example from a few weeks ago

> Imagine a hypothetical simplistic example, where we point our telescope to observe the light from a single star with true flux $F_{\rm true}$, which remains constant with time. We will ignore any systematic errors. We perform $N$ measurements with our telescope, where the $i^{\rm th}$ measurement report an observed flux $F_i$ with uncertainty/error $e_i$.

In [ ]:
# generate synthetic observations
F_true = 1000
F_obs = np.random.poisson(lam=F_true)

Let's write down the a simulation of the data generation process. In this case this is trivial because we know exactly the data generation process

In [ ]:
def simulation(F_true): 
    return np.random.poisson(lam=F_true)

For our prior, lets use a simple Gaussian
$$p(\theta) = \mathcal{N}(1002, 4)$$
This would hypothetically be measurement from a prior experiment

In [ ]:
# p(theta) = N(1002, 4)
def sample_prior(): 
    return 1002. + 4. *np.random.normal()

Then by repeating the following two steps,
1. sample the prior: $\theta'\sim p(\theta)$
2. run simulation: $\mathcal{D}' \sim F(\theta')$

We can sample $(\theta', \mathcal{D}') \sim p(\theta, \mathcal{D})$

In [ ]:
# now draw Nsample samples
Nsample = 10000

thetas, data = [], [] 
for i in range(Nsample):
    theta_p = sample_prior()
    data_p = simulation(theta_p)

    thetas.append(theta_p)
    data.append(data_p)

samples = np.array([thetas, data]).T

In [ ]:
# plot p(theta, D)
fig = plt.figure(figsize=(3,3))
sub = fig.add_subplot(111)
sub.scatter(np.array(thetas), np.array(data), s=1, c='k')
sub.set_xlabel(r'$\theta$', fontsize=25)
sub.set_ylabel('$\mathcal{D}$', fontsize=25)

## variational inference 
Now lets use variational inference to estimate this joint distribution (Lecture 18). First, we need to select our surrogate distribution $q_\phi(\theta, \mathcal{D})$. We'll use a simple 2D Gaussian, so the hyperparameter $\phi$ is the mean and covariance matrix

In [ ]:
def log_q(x, mean, cov):
    # log q_phi
    diff = x - mean               

    # cholseky factorization to lower-triangular matrix 
    L = torch.linalg.cholesky(cov) 
    
    # triangular solve trick so we don't 
    # have to invert the covariance matrix 
    sol = torch.linalg.solve_triangular(
        L, diff.T, upper=False).T

    # log det --- det(Cov) = det(L)^2
    logdet = 2.0 * torch.sum(torch.log(torch.diag(L)))

    D = mean.shape[-1]
    return -0.5 * (D * torch.log(torch.tensor(2 * torch.pi)) + logdet
                   + torch.sum(sol ** 2, dim=-1)) 

Now lets determine $\phi$, the mean and covariance, by minimizing the KL divergence between $p(\theta, \mathcal{D})$ and $q_\phi(\theta, \mathcal{D})$: 

$$D_{\rm KL} (p\parallel q_\phi) \approx \sum\limits_{(\theta',\mathcal{D}')\sim p(\theta, \mathcal{D})} \left[ - \log q_\phi(\theta',\mathcal{D}')\right]$$

Again, note that we're estimating the KL divergence using $(\theta',\mathcal{D}')\sim p(\theta, \mathcal{D})$ --- i.e. simulation-based inference!

In [ ]:
def KL(x, mu, A):
    '''KL divergence'''
    # some more common linear algebra tricks
    # A @ A.T ensure cov is positive semi-definite
    # 1e-3 * I ensures its symmetric positive definite
    cov = A @ A.T + 1e-3 * torch.eye(A.shape[0], device=A.device)
    return -torch.mean(log_q(x, mu, cov))

initialize $\phi$ and optimize using `Adam`

In [ ]:
mu = torch.tensor([1002.0, 1002.0], requires_grad=True)
A  = torch.tensor([[4.0, 0.0], [0.0, 30.0]], requires_grad=True)

In [ ]:
optimizer = torch.optim.Adam([mu, A], lr=0.01)

In [ ]:
for t in range(2001):
    optimizer.zero_grad()
    loss = KL(torch.tensor(samples), mu, A)
    loss.backward()
    optimizer.step()

    if t % 200 == 0:
        cov = (A @ A.T + 1e-3 * torch.eye(2)).detach()
        print(f"step {t:4d} | loss {loss.item():.4f} | "
              f"mu {mu.detach().numpy()} | "
              f"cov diag [{cov[0,0]:.2f}, {cov[1,1]:.2f}]")

Lets check how well $q_\phi(\theta, \mathcal{D})$ approximates $p(\theta, \mathcal{D})$

In [ ]:
q_samples = np.random.multivariate_normal(
    np.array(mu.detach().numpy()), 
    np.array(A.detach().numpy() @ A.detach().numpy().T), size=100000)

In [ ]:
fig = plt.figure(figsize=(4,4))
fig = corner.corner(np.array(samples), hist_kwargs={"density": True}, fig=fig)
fig = corner.corner(q_samples, color='C1', hist_kwargs={"density": True}, fig=fig)

looks pretty good!

Now we can move onto ABC using the surrogate $q_\phi$. The advantage here is that $q_\phi$ is extremely cheap to sample, so we can generate millions of samples that pass our distance critiera

In [ ]:
q_samples = np.random.multivariate_normal(
    np.array(mu.detach().numpy()), 
    np.array(A.detach().numpy() @ A.detach().numpy().T), size=10000000)
close_enough = np.abs(q_samples[:,1] - F_obs) < 1
print('%i have D \\approx D_obs' % np.sum(close_enough))

In [ ]:
plt.figure(figsize=(4,2))
plt.hist(q_samples[:,0][close_enough], range=(960, 1040), bins=40)
plt.axvline(F_true, color='k', linestyle='--')
plt.xlabel(r'$F_{\rm true}$', fontsize=15); plt.show()

This is a bit of a silly example since the simulator is very cheap to run. But in many astrophysical applications the simulator of the data generation process is very expensive (cosmological simulations), then it would make sense to sample the joint distribution using the surrogate!

# Example 1
Now that was a very simple 1D example. Lets consider a slightly more interesting example of fitting a line to some data and see how SBI with variational inference holds up.

First, we'll go through the example with a slight modification: Gaussian priors rather than uniform priors. 

In [ ]:
a_true, b_true = 0, 1.75
x = np.array([1, 1.5, 2, 2.5])
y = np.array([1.87640523, 2.59001572, 3.4978738 , 4.47408932])
yerr = 0.1

In [ ]:
plt.figure(figsize=(4,3))
plt.errorbar(x, y, yerr=yerr, fmt='.k')

same as above, sample $\theta', \mathcal{D}' \sim p(\theta, \mathcal{D})$ using prior and simulator

In [ ]:
def simulation(theta): 
    a, b = theta
    noise = yerr * np.random.normal(size=len(x))
    return a + b * x + noise

def sample_prior(): 
    # sample a and b from uniform prior
    _a = -0.5 + np.random.normal() 
    _b = 5 + 3*np.random.normal()
    return [_a, _b]

In [ ]:
Nsample = 50000

samples = np.zeros((Nsample, 6))
for i in range(Nsample): 
    theta_p = sample_prior()
    data_p = simulation(theta_p)
    samples[i,:] = np.concatenate([theta_p, data_p])

Initialize and optimize $\phi$. We can use the `log_q` and `KL` functions from earlier. 

In [ ]:
mu = torch.tensor([0., 2., 5., 5., 10., 10.], requires_grad=True)
A  = torch.diag(torch.tensor([1, 3., 3., 4., 6, 7.5])).requires_grad_(True)

In [ ]:
optimizer = torch.optim.Adam([mu, A], lr=0.01)

In [ ]:
for t in range(2001):
    optimizer.zero_grad()
    loss = KL(torch.tensor(samples), mu, A)
    loss.backward()
    optimizer.step()

    if t % 200 == 0:
        cov = (A @ A.T + 1e-3 * torch.eye(6)).detach()
        print(f"step {t:4d} | loss {loss.item():.4f} | ")

Lets again check $q_\phi(\theta, \mathcal{D})$ against $p(\theta, \mathcal{D})$

In [ ]:
q_samples = np.random.multivariate_normal(
    np.array(mu.detach().numpy()), 
    np.array(A.detach().numpy() @ A.detach().numpy().T) + 1e-3 * np.eye(6), 
    size=100000)

In [ ]:
fig = plt.figure(figsize=(4,4))
fig = corner.corner(np.array(samples), hist_kwargs={'density': True}, fig=fig)
fig = corner.corner(q_samples, color='C1', hist_kwargs={'density': True}, fig=fig)

comparison of $q_\phi$ to $p$ looks pretty good. Let's move on to the ABC approximation

In [ ]:
q_samples = np.random.multivariate_normal(
    np.array(mu.detach().numpy()), 
    np.array(A.detach().numpy() @ A.detach().numpy().T) + 1e-3 * np.eye(6), 
    size=10000000)

In [ ]:
# ABC using distance criteria rho(D, D_obs) < epsilon
rho = np.sqrt(np.sum((q_samples[:,2:] - y[None, :])**2, axis=1))

In [ ]:
epsilon = 0.2
print('reject %f percent of simulations' % (100.*(1-np.mean(rho < epsilon))))

Because we using a surrogate that is very easy to sample, we can afford to reject 99.95% of the samples!

In [ ]:
fig = plt.figure(figsize=(4,4))
fig = corner.corner(q_samples[:,:2][rho < epsilon], labels=['a', 'b'], fig=fig)
corner.overplot_lines(fig, [a_true, b_true], color="C1")
corner.overplot_points(fig, np.array([a_true, b_true])[None], marker="s", color="C1")

posterior looks pretty good!

# Example 1.b
Now what happens if we change the prior back to the uniform priors, instead of the Gaussian priors

In [ ]:
def simulation(theta): 
    a, b = theta
    noise = yerr * np.random.normal(size=len(x))
    return a + b * x + noise

def sample_prior(): 
    # sample a and b from uniform prior
    _a = -0.5 + np.random.uniform() 
    _b = 10*np.random.uniform()
    return [_a, _b]

Nsample = 500000

samples = np.zeros((Nsample, 6))
for i in range(Nsample): 
    theta_p = sample_prior()
    data_p = simulation(theta_p)
    samples[i,:] = np.concatenate([theta_p, data_p])

In [ ]:
mu = torch.tensor([0., 2., 5., 5., 10., 10.], requires_grad=True)
A  = torch.diag(torch.tensor([1, 3., 3., 4., 6, 7.5])).requires_grad_(True)

optimizer = torch.optim.Adam([mu, A], lr=0.01)

for t in range(2001):
    optimizer.zero_grad()
    loss = KL(torch.tensor(samples), mu, A)
    loss.backward()
    optimizer.step()

    if t % 200 == 0:
        cov = (A @ A.T + 1e-3 * torch.eye(6)).detach()
        print(f"step {t:4d} | loss {loss.item():.4f} | ")

In [ ]:
q_samples = np.random.multivariate_normal(
    np.array(mu.detach().numpy()), 
    np.array(A.detach().numpy() @ A.detach().numpy().T) + 1e-3 * np.eye(6), 
    size=100000)
fig = plt.figure(figsize=(4,4))
fig = corner.corner(np.array(samples), hist_kwargs={'density': True}, fig=fig)
fig = corner.corner(q_samples, color='C1', hist_kwargs={'density': True}, fig=fig)

Now you can see why I chose Gaussian priors earlier.

With uniform priors, we're confronted with a key challenge with variational inference: 
> Our multivariate Gaussian surrogate cannot estimate $p(\theta, \mathcal{D})$ accurately. In this case, there's no way to accurately estimate the posterior with $q_\phi$.

Since $q_\phi$ does not accurately estimate $p(\theta, \mathcal{D})$, we cannot proceed with the Bayesian inference

# In-class exercise: the IMF slope with a surrogate

In the Lecture 19 exercise, we inferred the slope $\alpha$ of the initial mass function from the masses of 100 stars with ABC, using 200,000 simulated clusters. 
That was affordable only because the simulator was a one-liner. 
What if every simulated cluster is expensive, a star formation simulation that takes hours, and we only have the computation budget for **1000 simulations**. 

With 1000 simulations, ABC has to keep a large $\epsilon$ and doesn't converge to the true posterior. Instead, let's use SBI + VI 

1. sample $\{(\theta', \mathcal{D}')\}\sim p(\theta, \mathcal{D})$ using the prior and the simulator $F$;
2. select $q_\phi(\theta, \mathcal{D})$ and optimize $\phi$ by minimizing $\sum[-\log q_\phi(\theta', \mathcal{D}')]$;
3. run ABC on samples from the cheap surrogate, $(\theta', \mathcal{D}')\sim q_\phi(\theta, \mathcal{D})$.

As in Lecture 19, the likelihood of this problem is known so we can check our estimated posteriors against the exact posterior.

### simulator, summaries, prior
The simulator is the same as in the last lecture: masses from $p(m\,|\,\alpha)\propto m^{-\alpha}$ between 1 and 20 $M_\odot$.
As data $\mathcal{D}$ we use two summaries of the 100 masses: the mean log mass and the log of the standard deviation of the log masses. So $(\theta, \mathcal{D})$ is 3-dimensional.

For the prior, suppose earlier studies of other young clusters give $\alpha \sim \mathcal{N}(\mu = 2.35, \sigma = 0.2)$. The cluster we observe has $\alpha_{\rm true} = 2.2$.

In [ ]:
rng = np.random.default_rng(20)
torch.manual_seed(20)

M_MIN, M_MAX, N_STAR = 1.0, 20.0, 100

def simulate_masses(alpha, rng, n=N_STAR):
    ''' One simulated cluster of n stellar masses per value of alpha.
    '''
    alpha = np.atleast_1d(alpha)[:, None]
    e = 1 - alpha
    u = rng.uniform(size=(alpha.shape[0], n))
    return (M_MIN**e + u * (M_MAX**e - M_MIN**e))**(1 / e)

def summaries(m):
    ''' D = (mean log mass, log std of log mass) for each row of masses.
    '''
    logm = np.log(m)
    return np.column_stack([logm.mean(1), np.log(logm.std(1))])

# prior
PRIOR_MU, PRIOR_SD = 2.35, 0.2
sample_prior = lambda n: rng.normal(PRIOR_MU, PRIOR_SD, n)
log_prior = lambda a: -0.5 * ((a - PRIOR_MU) / PRIOR_SD)**2

def exact_posterior(m, log_prior, a_grid=np.linspace(1.2, 3.8, 4001)):
    ''' Exact posterior of alpha on given grid.
    '''
    e = 1 - a_grid
    lp = len(m) * np.log(e / (M_MAX**e - M_MIN**e)) - a_grid * np.log(m).sum() + log_prior(a_grid)
    p = np.exp(lp - lp.max())
    p /= np.trapezoid(p, a_grid)
    return a_grid, p

alpha_true = 2.2
m_obs = simulate_masses(alpha_true, rng)[0]
D_obs = summaries(m_obs[None])[0]
a_grid, post_exact = exact_posterior(m_obs, log_prior)
mu_exact = np.trapezoid(a_grid * post_exact, a_grid)
sd_exact = np.sqrt(np.trapezoid((a_grid - mu_exact)**2 * post_exact, a_grid))
print('D_obs =', D_obs.round(4))
print(f'exact posterior: alpha = {mu_exact:.3f} +/- {sd_exact:.3f}')

### 1. Sample $p(\theta, \mathcal{D})$ with the simulator
Draw 1000 $(\alpha', \mathcal{D}')$ pairs using the prior and the simulator, and plot them with `corner`.

In [ ]:
N_SIM = 1000

### 2. Select $q_\phi(\theta, \mathcal{D})$ and optimize $\phi$
Fit a 3D Gaussian $q_\phi$ by minimizing the KL divergence. You can use the `log_q` and `KL` functions from the examples above.

*Hint: For a Gaussian $q_\phi$ its minimum is known analytically: the sample mean and sample covariance of the simulations.*

Compare samples from $q_\phi$ with the simulations.

### 3. ABC with the surrogate $q_\phi$

Draw $10^7$ samples from $q_\phi$ and perfrom ABC$^{[1]}$. 
Compare the resulting posterior with the exact posterior.
Then compare with what rejection ABC can do with the original 1000 simulations$^{[2]}$


$^{[1]}$: For $\rho(\mathcal{D}', \mathcal{D}_{\rm obs}) < \epsilon$, the two dimensions of the summaries have different ranges. Measure the distance in units of each summary's standard deviation over the simulations. 

$^{[2]}$: Use $\epsilon$ that keeps the ~50 closest to $\mathcal{D}_{\rm obs}$.

### 4. Amortization
The surrogate $q_\phi(\theta, \mathcal{D})$ describes the joint distribution for *any* data set, not just $\mathcal{D}_{\rm obs}$. Say we observed three more clusters with: $\alpha_{\rm true}$ = 1.9, 2.35 and 2.7. 
Infer their slopes without running any new simulations and compare to the exact posterior.

In [ ]:
print(' alpha_true |  surrogate ABC   |      exact')
for a_t in [1.9, 2.35, 2.7]:
    # new observation (not a training simulation)
    m_new = simulate_masses(a_t, rng)[0]


### (Optional) When the Gaussian surrogate breaks
Repeat steps 1–3 with a broad uniform prior, $\alpha\sim\mathcal{U}(1.5, 3.5)$ (same as Lecture 19). 
Exmaine the joint distribution. Then infer $\alpha$ for clusters with $\alpha_{\rm true}$ = 1.7, 2.2 and 3.2. How does the estimated posterior compare to the true posterior? 